Colab is making it easier than ever to integrate powerful Generative AI capabilities into your projects. We are launching public preview for a simple and intuitive Python library (google.colab.ai) to access state-of-the-art language models directly within Colab environments. All users have free access to most popular LLMs, while paid users have access to a wider selection of models. This means users can spend less time on configuration and set up and more time bringing their ideas to life. With just a few lines of code, you can now perform a variety of tasks:
- Generate text
- Translate languages
- Write creative content
- Categorize text

Happy Coding!


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/Getting_started_with_google_colab_ai.ipynb)

In [1]:
# @title List available models
from google.colab import ai

ai.list_models()

['google/gemini-2.0-flash',
 'google/gemini-2.0-flash-lite',
 'google/gemini-2.5-flash',
 'google/gemini-2.5-flash-lite',
 'google/gemini-2.5-pro',
 'google/gemma-3-12b',
 'google/gemma-3-1b',
 'google/gemma-3-27b',
 'google/gemma-3-4b']

Choosing a Model
The model names give you a hint about their capabilities and intended use:

Pro: These are the most capable models, ideal for complex reasoning, creative tasks, and detailed analysis.

Flash: These models are optimized for high speed and efficiency, making them great for summarization, chat applications, and tasks requiring rapid responses.

Gemma: These are lightweight, open-weight models suitable for a variety of text generation tasks and are great for experimentation.

In [2]:
# @title Simple batch generation example
# Only text-to-text input/output is supported
from google.colab import ai

response = ai.generate_text("What is the capital of France?")
print(response)

The capital of France is **Paris**.


In [3]:
# @title Choose a different model
from google.colab import ai

response = ai.generate_text("What is the capital of England", model_name='google/gemini-2.0-flash-lite')
print(response)

The capital of England is **London**.



For longer text generations, you can stream the response. This displays the output token by token as it's generated, rather than waiting for the entire response to complete. This provides a more interactive and responsive experience. To enable this, simply set stream=True.

In [4]:
#@title Text formatting setup
#code is not necessary for colab.ai, but is useful in fomatting text chunks
import sys

class LineWrapper:
    def __init__(self, max_length=80):
        self.max_length = max_length
        self.current_line_length = 0

    def print(self, text_chunk):
        i = 0
        n = len(text_chunk)
        while i < n:
            start_index = i
            while i < n and text_chunk[i] not in ' \n': # Find end of word
                i += 1
            current_word = text_chunk[start_index:i]

            delimiter = ""
            if i < n: # If not end of chunk, we found a delimiter
                delimiter = text_chunk[i]
                i += 1 # Consume delimiter

            if current_word:
                needs_leading_space = (self.current_line_length > 0)

                # Case 1: Word itself is too long for a line (must be broken)
                if len(current_word) > self.max_length:
                    if needs_leading_space: # Newline if current line has content
                        sys.stdout.write('\n')
                        self.current_line_length = 0
                    for char_val in current_word: # Break the long word
                        if self.current_line_length >= self.max_length:
                            sys.stdout.write('\n')
                            self.current_line_length = 0
                        sys.stdout.write(char_val)
                        self.current_line_length += 1
                # Case 2: Word doesn't fit on current line (print on new line)
                elif self.current_line_length + (1 if needs_leading_space else 0) + len(current_word) > self.max_length:
                    sys.stdout.write('\n')
                    sys.stdout.write(current_word)
                    self.current_line_length = len(current_word)
                # Case 3: Word fits on current line
                else:
                    if needs_leading_space:
                        # Define punctuation that should not have a leading space
                        # when they form an entire "word" (token) following another word.
                        no_leading_space_punctuation = {
                            ",", ".", ";", ":", "!", "?",        # Standard sentence punctuation
                            ")", "]", "}",                     # Closing brackets
                            "'s", "'S", "'re", "'RE", "'ve", "'VE", # Common contractions
                            "'m", "'M", "'ll", "'LL", "'d", "'D",
                            "n't", "N'T",
                            "...", "…"                          # Ellipses
                        }
                        if current_word not in no_leading_space_punctuation:
                            sys.stdout.write(' ')
                            self.current_line_length += 1
                    sys.stdout.write(current_word)
                    self.current_line_length += len(current_word)

            if delimiter == '\n':
                sys.stdout.write('\n')
                self.current_line_length = 0
            elif delimiter == ' ':
                # If line is full and a space delimiter arrives, it implies a wrap.
                if self.current_line_length >= self.max_length:
                    sys.stdout.write('\n')
                    self.current_line_length = 0

        sys.stdout.flush()


In [5]:
import cadquery as cq

# ---------- parametric constants ----------
# (match your spec § 4.2)
D_cham   = 28 * 25.4          # 28"  → mm
H_cham   = 36 * 25.4          # 36"
D_cap    = 30 * 25.4          # 30"
T_cap    = 4  * 25.4          # 4"
D_shaft  = 2  * 25.4          # 2"
N_elec   = 8
Ele_W    = 8  * 25.4
Ele_H    = 24 * 25.4
Ele_T    = 0.25 * 25.4
Vent_D   = 1.5 * 25.4         # inlet Ø
Vent_ang = 45                 # degrees
explode  = 30                 # visual gap between stages

# ---------- helpers ----------
def ring_coords(n, r):
    from math import cos, sin, tau
    return [(r * cos(i * tau / n), r * sin(i * tau / n)) for i in range(n)]

# ---------- 1. UPPER CAP ----------
upper_cap = (cq.Workplane()
             .circle(D_cap/2).extrude(T_cap)
             .faces(">Z").workplane()
             .hole(D_shaft)                 # central shaft hole
             .faces(">Z")
             .rect(D_cap-20, D_cap-20, forConstruction=True)
             .vertices().hole(6)             # 8× bolt holes
            )

# ---------- 2. ELECTRODE PLATES (8×) ----------
ele_pts = ring_coords(N_elec, D_cham/2 - 15)
electrodes = [(cq.Workplane()
               .center(*p).rect(Ele_W, Ele_T).extrude(Ele_H))
              for p in ele_pts]

# ---------- 3. VENTURI TUBES (10×) ----------
vent_pts = ring_coords(10, D_cham/2 + explode/2)
venturis = [(cq.Workplane()
             .center(*p).workplane(offset=-explode/2)
             .circle(Vent_D/2).extrude(explode))
            for p in vent_pts]

# ---------- 4. CHAMBER BODY ----------
body = (cq.Workplane()
        .circle(D_cham/2).extrude(H_cham)
        .faces(">Z").workplane().hole(D_shaft)
        # cut venturi ports
        .faces(">Z").workplane()
        .pushPoints(ring_coords(10, D_cham/2))
        .circle(Vent_D/2).cutThruAll()
       )

# ---------- 5. IMPELLER ----------
hub = cq.Workplane().circle(D_shaft/2+10).extrude(20)
blades = [(cq.Workplane()
           .center(D_shaft/2+5, 0).rect(40, 3).extrude(15)
           .rotate((0,0,0), (0,0,1), i*360/4))
          for i in range(4)]
impeller = hub.union(*blades)

# ---------- 6. LOWER CAP ----------
lower_cap = upper_cap.mirror("XY")  # same geometry, mirrored

# ---------- 7. exploded assembly ----------
assy = cq.Assembly()
assy.add(upper_cap, loc=cq.Location(cq.Vector(0, 0, H_cham + explode*2)))
assy.add(body,       loc=cq.Location(cq.Vector(0, 0, explode)))
assy.add(lower_cap,  loc=cq.Location(cq.Vector(0, 0, -explode)))
for i, e in enumerate(electrodes):
    assy.add(e, loc=cq.Location(cq.Vector(0, 0, explode*1.5),
                                cq.Vector(0, 0, 1), i*360/N_elec))
assy.add(impeller,   loc=cq.Location(cq.Vector(0, 0, -explode*1.5)))
for i, v in enumerate(venturis):
    assy.add(v, loc=cq.Location(cq.Vector(0, 0, 0),
                                cq.Vector(0, 0, 1), i*360/10))

# ---------- 8. export ----------
# in CQ-editor: select assy → Export SVG → inkscape → PDF
show_object(assy)

ModuleNotFoundError: No module named 'cadquery'